In [1]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

print("Total number of Character:", len(raw_text))
print(raw_text[:99])

Total number of Character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
import re
text = "Hello, world. This, is a test."
result = re.split(r'(\s)',text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [3]:
result = re.split(r'([,.]|\s)',text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [4]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [5]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"() \']|--|\\s)',text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [6]:
preprocessed = re.split(r'([,.:;?_!"() \']|--|\\s)', raw_text)
result = [item.strip() for item in preprocessed if item.strip()]
print(len(result))

4690


In [7]:
print(result[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into token IDs

In [8]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1149


In [9]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i,item in enumerate(vocab.items()):
    print(item)
    if i>=50:
        break

('', 0)
('\n\n', 1)
('\n\nA', 2)
('\n\nAnd', 3)
('\n\nAs', 4)
('\n\nBut', 5)
('\n\nFor', 6)
('\n\nGisburn', 7)
('\n\nHe', 8)
('\n\nHis', 9)
('\n\nI', 10)
('\n\nIn', 11)
('\n\nIt', 12)
('\n\nMrs', 13)
('\n\nOf', 14)
('\n\nPoor', 15)
('\n\nShe', 16)
('\n\nThe', 17)
('\n\nWell', 18)
('\n\nYes', 19)
(' ', 20)
('!', 21)
('"', 22)
("'", 23)
('(', 24)
(')', 25)
(',', 26)
('--', 27)
('.', 28)
(':', 29)
(';', 30)
('?', 31)
('A', 32)
('Ah', 33)
('Among', 34)
('And', 35)
('Are', 36)
('Arrt', 37)
('At', 38)
('Be', 39)
('Begin', 40)
('Burlington', 41)
('But', 42)
('By', 43)
('Carlo', 44)
('Chicago', 45)
('Claude', 46)
('Come', 47)
('Croft', 48)
('Destroyed', 49)
('Devonshire', 50)


In [10]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab   #1
        self.int_to_str= {i:s for s,i in vocab.items()}   #2

    def encode(self,text):  #3
        preprocessed = re.split(r'([,.:;?_!"() \']|--|\\s)', text)
        result = [item.strip() for item in preprocessed if item.strip()]  
        ids = [self.str_to_int[s] for s in result]
        return ids

    def decode(self,ids):   #4
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+(,.?!"()\\)', r"\\1",text)  #5
        return text
        

In [59]:
#1 Stores the vocabulary as class attribute for access in encode and decode
#2 creates an inverse vocabulary that maps token IDs back to texts
#3 processes input text to token IDs
#4 Converts token IDs back into text
#5 remove spaces before specified punctuation

In [11]:
tokenizer = SimpleTokenizerV1(vocab)
text =""""It's the last he painted, you know,"Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[22, 75, 23, 869, 1007, 621, 552, 765, 26, 1145, 615, 26, 22, 86, 28, 58, 870, 1127, 773, 812, 28]


In [12]:
print(tokenizer.decode(ids))

" It ' s the last he painted , you know , " Mrs . Gisburn said with pardonable pride .


## Adding Special Context tokens

In [13]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}

print(len(vocab.items()))

1151


In [14]:
for i,item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1146)
('your', 1147)
('yourself', 1148)
('<|endoftext|>', 1149)
('<|unk|>', 1150)


In [15]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int=vocab   #1
        self.int_to_str= {i:s for s,i in vocab.items()}   #2

    def encode(self,text):  #3
        preprocessed = re.split(r'([,.:;?_!"() \']|--|\\s)', text)
        result = [item.strip() for item in preprocessed if item.strip()] 
        result = [item if item in self.str_to_int else "<|unk|>" for item in result]
        ids = [self.str_to_int[s] for s in result]
        return ids

    def decode(self,ids):   #4
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+(,.?!"()\\)', r"\\1",text)  #5
        return text
        

In [16]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palaces."
text = " <|endoftext|> ".join((text1,text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palaces.


In [17]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1150, 26, 374, 1145, 647, 994, 31, 1149, 1150, 1007, 975, 1003, 741, 1007, 1150, 28]


In [18]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|> , do you like tea ? <|endoftext|> <|unk|> the sunlit terraces of the <|unk|> .


### vocabulary is basically collection of all unique words in your training data. each word in vocabulary will be assigned an integer so that when you input a text, you can convert that input text into token IDs and then back to text as well. if any word from your input text is not part of vocab then we need to assign a token 'unk' to it. when you train model like GPT we also add words like 'endoftext' to tell GPT that this is end of one article or book or data source.

## Byte pair encoding

In [87]:
!pip install tiktoken


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from importlib.metadata import version
import tiktoken
print("tiktoken version: ",version("tiktoken"))

tiktoken version:  0.12.0


In [20]:
tokenizer = tiktoken.get_encoding("gpt2")

In [21]:
text = ("Hello, do you like tea? <|endoftext>| In the sunlit terraces of someunknownpalace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext>|"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 1279, 91, 437, 1659, 5239, 29, 91, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 18596, 558, 13]


In [22]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext>| In the sunlit terraces of someunknownpalace.


## Data Smapling with a sliding window

In [23]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [24]:
enc_sample = enc_text[50:]

In [25]:
context_size = 4
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:     {y}")

x: [290, 4920, 2241, 287]
y:     [4920, 2241, 287, 257]


In [26]:
for i in range(1,context_size+1):
    context=enc_sample[:i]
    desired=enc_sample[i]
    print(context,"---->",desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [28]:
for i in range(1,context_size+1):
    context=enc_sample[:i]
    desired=enc_sample[i]
    print(tokenizer.decode(context),"---->",tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [31]:
import torch
from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        token_ids = tokenizer.encode(txt) #1

        for i in range(0,len(token_ids)-max_length,stride): #2
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self): #3
        return len(self.input_ids)

    def __getitem__(self,idx): #4
        return self.input_ids[idx],self.target_ids[idx]
        

#1 Tokenizes the entire text
#2 uses a sliding window to chunk the book into overlapping sequences of max_length
#3 Returns the total number of rows in the dataset
#4 Returns a single row from the dataset

In [32]:
def create_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)
    dataloader=DataLoader(dataset,
    batch_size=batch_size,
    shuffle=shuffle,
    drop_last=drop_last,
    num_workers=num_workers)

    return dataloader

In [39]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

dataloader=create_dataloader_v1(raw_text,batch_size=1,max_length=4, stride=1,shuffle=False)
data_iter=iter(dataloader) #
first_batch=next(data_iter)
print(first_batch)

# Converts dataloader into a Python iterator to fetch the next entry  via Python's built-in next() function

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [43]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

dataloader=create_dataloader_v1(raw_text,batch_size=2,max_length=4, stride=1,shuffle=False)
data_iter=iter(dataloader) #
first_batch=next(data_iter)
print(first_batch)

# Converts dataloader into a Python iterator to fetch the next entry  via Python's built-in next() function

[tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807]]), tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619]])]


In PyTorch, the batch_size argument in the DataLoader determines how many data samples are grouped together into a single tensor (a "mini-batch") for one forward and backward pass during training. It is a crucial hyperparameter for balancing training speed, memory consumption, and model generalization. 
Key Aspects of batch_size in PyTorch:
Default Behavior: By default, batch_size is set to 1.
Automatic Batching: When a batch_size is specified (e.g., 64), the DataLoader automatically collates individual samples from the Dataset into a single, stacked tensor.
Common Values: Typical values range between 32 and 256.
Performance Impact:
Small Batch Sizes (e.g., 16-32): Provide noisier gradient estimates, which can help escape local minima and improve generalization, but may be slower to train.
Large Batch Sizes (e.g., 256-512): Offer more accurate gradients and faster training due to efficient GPU utilization, but may require more memory and can lead to poorer generalization. 
Important Considerations:
Memory Constraints: Larger batch sizes require more VRAM. If you encounter out-of-memory errors, decrease the batch_size.
Last Batch: If the total dataset size is not divisible by the batch_size, the final batch will be smaller than the rest. You can set drop_last=True in the DataLoader to discard this last, smaller batch.
Hardware Efficiency: Power-of-two values (e.g., 32, 64, 128, 256) are recommended because they are optimized for GPU computation.
Learning Rate Scaling: If you significantly increase the batch size, you may need to increase the learning rate to compensate. 

In [40]:
second_batch=next(data_iter)
print(second_batch)
# Stride of 1 will shift the input by 1

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


# Creating Token embeddings

In [45]:
#suppose we have following four input tokens with IDs 2,3,5,1
input_ids = torch.tensor([2,3,5,1])
vocab_size=6
output_dim=3

In [46]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
print(embedding_layer.weight)


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [49]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [50]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## Encoding Word Positions

#### Absolute positional embeddings give tokens fixed, unique IDs (like house numbers) for their exact spot in a sequence, added to inputs; while relative positional embeddings focus on the distance between tokens (e.g., 3 steps away), often modifying the attention mechanism for better generalization to varying lengths

In [53]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [54]:
max_length = 4
dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=max_length,stride=max_length,shuffle=False)
data_iter = iter(dataloader)
inputs,targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs Shape:\\n",inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs Shape:\n torch.Size([8, 4])


In [55]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [56]:
# 8 inputs with 4 tokens each with each token having embedding of size 256

In [57]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length,output_dim)
pos_embedding = pos_embedding_layer(torch.arange(context_length))
print(pos_embedding.shape)

torch.Size([4, 256])


In [58]:
input_embeddings = token_embeddings + pos_embedding
print(input_embeddings.shape)

torch.Size([8, 4, 256])
